In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier,plot_tree
import matplotlib.pyplot as plt

In [7]:
df = pd.read_csv("clean_disease_dataset.csv")
print(df.shape)
print(df.head())
print(df.shape)

(16, 61)
   vomiting  abdominal_pain  anxious  bleeding_from_the_nose_or_gums  \
0         0               0        0                               0   
1         0               0        0                               0   
2         0               0        0                               0   
3         0               0        0                               0   
4         0               0        0                               1   

   blurry_vision  changing_moods_or_behaviors  chest_pain  migraine  coughing  \
0              0                            0           0         0         0   
1              0                            0           0         0         1   
2              0                            0           1         0         0   
3              0                            0           0         0         0   
4              0                            0           0         0         0   

   coughing_up_blood_or_sputum  ...  unexplained_mood_changes  \
0     

In [8]:
df.columns

Index(['vomiting', 'abdominal_pain', 'anxious',
       'bleeding_from_the_nose_or_gums', 'blurry_vision',
       'changing_moods_or_behaviors', 'chest_pain', 'migraine', 'coughing',
       'coughing_up_blood_or_sputum', 'cramps', 'dark_yellow_urine',
       'depression', 'diarrhea', 'had_dengue_before', 'fatigue',
       'feeling_very_thirsty', 'fever', 'gray_or_clay-colored_stools',
       'having_thoughts_of_death_or_suicide', 'headaches', 'hepatitis_b',
       'hepatitis_c', 'increased_sensitivity_to_light',
       'increasing_use_of_alcohol_or_drugs', 'internal_bleeding', 'irritable',
       'jaundice', 'joint_pain', 'losing_weight_without_trying',
       'loss_of_appetite', 'have_tb_germs_in_their_bodies',
       'muscle_or_body_aches', 'nausea', 'night_sweats',
       'numbness_or_tingling_in_the_feet', 'restless', 'blood_in_the_stool',
       'digestive_problems', 'irritable.1', 'sneezing', 'overeating', 'rash',
       'runny_or_stuffy_nose', 'severe_abdominal_pain', 'shortness_

In [9]:
print(df['disease'].value_counts())

disease
Influenza        1
COVID-19         1
Tuberculosis     1
Malaria          1
Dengue fever     1
Chickenpox       1
Measles          1
Typhoid fever    1
Hepatitis B      1
Hepatitis C      1
Asthma           1
Pneumonia        1
Hypertension     1
Diabetes         1
Migraine         1
Depression       1
Name: count, dtype: int64


In [10]:
import random
import pandas as pd

final_df = df

def augment_data(df, n_samples=10):
    augmented = []

    for _, row in df.iterrows():
        disease = row["disease"]
        
        symptoms = row.drop("disease")
        active_symptoms = [col for col in symptoms.index if symptoms[col] == 1]

        for _ in range(n_samples):
            new_row = symptoms.copy()

            # 🔴 FIX 1: handle empty case
            if len(active_symptoms) == 0:
                new_row["disease"] = disease
                augmented.append(new_row)
                continue

            # 🔴 FIX 2: safe drop count
            max_drop = len(active_symptoms) // 2
            drop_count = random.randint(0, max_drop)

            # 🔴 FIX 3: safe sampling
            drop_symptoms = random.sample(active_symptoms, drop_count)

            for sym in drop_symptoms:
                new_row[sym] = 0

            new_row["disease"] = disease
            augmented.append(new_row)

    return pd.DataFrame(augmented)

In [28]:
# FIRST augment 
aug_df = augment_data(final_df, n_samples=10) 
aug_df=aug_df.drop("Unnamed: 57", axis=1)
# THEN split 
from sklearn.model_selection import train_test_split 
X = aug_df.drop("disease", axis=1) 
#X = aug_df.drop("Unnamed: 57", axis=1)
y = aug_df["disease"] 
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, stratify=y, random_state=42 )

In [29]:
print(X)
X.columns

    vomiting  abdominal_pain  anxious  bleeding_from_the_nose_or_gums  \
0          0               0        0                               0   
0          0               0        0                               0   
0          0               0        0                               0   
0          0               0        0                               0   
0          0               0        0                               0   
..       ...             ...      ...                             ...   
15         0               0        1                               0   
15         0               0        1                               0   
15         0               0        1                               0   
15         0               0        1                               0   
15         0               0        1                               0   

    blurry_vision  changing_moods_or_behaviors  chest_pain  migraine  \
0               0                            0     

Index(['vomiting', 'abdominal_pain', 'anxious',
       'bleeding_from_the_nose_or_gums', 'blurry_vision',
       'changing_moods_or_behaviors', 'chest_pain', 'migraine', 'coughing',
       'coughing_up_blood_or_sputum', 'cramps', 'dark_yellow_urine',
       'depression', 'diarrhea', 'had_dengue_before', 'fatigue',
       'feeling_very_thirsty', 'fever', 'gray_or_clay-colored_stools',
       'having_thoughts_of_death_or_suicide', 'headaches', 'hepatitis_b',
       'hepatitis_c', 'increased_sensitivity_to_light',
       'increasing_use_of_alcohol_or_drugs', 'internal_bleeding', 'irritable',
       'jaundice', 'joint_pain', 'losing_weight_without_trying',
       'loss_of_appetite', 'have_tb_germs_in_their_bodies',
       'muscle_or_body_aches', 'nausea', 'night_sweats',
       'numbness_or_tingling_in_the_feet', 'restless', 'blood_in_the_stool',
       'digestive_problems', 'irritable.1', 'sneezing', 'overeating', 'rash',
       'runny_or_stuffy_nose', 'severe_abdominal_pain', 'shortness_

In [30]:
from sklearn.preprocessing import LabelEncoder 
le=LabelEncoder()
y_train_numeric=le.fit_transform(y_train)
y_test_numeric=le.transform(y_test)
print(y_train_numeric)
print(y_test_numeric)
print(le.classes_)

[ 8  5  1  9  5 13  2  2 15 13 10  4  0  8  5 13  0  3 12  6  3 13  9 12
  4  1  4  2  4  5  2  0  8  7  1 13  7 14 11  3  9  2 10  4 11  6 13 10
  6 11  6  2  6 12 14  9  0  4  1  8  7  7 10  6 13 12 12  2 10 14  4 15
 15  9  8  8  0 14 10 15  0 12  9 11  3 10  5 11 13  0  7  7  5 15  8  3
 10  7  8  3  1  4  3  5 15 14  2 12 11  0 14  6 14 15 11  7  6  9  5 11
  1  9  1  3 12  1 15 14]
[11 10  6  9  3 10  2  1 15  7 15  6  4 14  5 11  8  2  5 12  0  0  8  9
 13 12  7 13  3  1  4 14]
['Asthma' 'COVID-19' 'Chickenpox' 'Dengue fever' 'Depression' 'Diabetes'
 'Hepatitis B' 'Hepatitis C' 'Hypertension' 'Influenza' 'Malaria'
 'Measles' 'Migraine' 'Pneumonia' 'Tuberculosis' 'Typhoid fever']


In [31]:
def calculate_accuracy(y_test,y_pred):
    print("\n Confusion Matrix \n", confusion_matrix(y_test,y_pred))
    print("\n Accuracy Score \n", accuracy_score(y_test, y_pred) * 100)
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [14]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [32]:
import pandas as pd

for col in X.columns:
    if not pd.api.types.is_numeric_dtype(X[col]):
        print(f"Column '{col}' is NOT numeric")

In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Store best models
best_models = {}

# 1. Logistic Regression
param_lr = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}
grid_lr = GridSearchCV(LogisticRegression(max_iter=1000), param_lr, cv=5, n_jobs=-1)
grid_lr.fit(X_train, y_train)
best_models['Logistic'] = grid_lr.best_estimator_

# 2. SVM
param_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']
}
grid_svm = GridSearchCV(SVC(probability=True), param_svm, cv=5, n_jobs=-1)
grid_svm.fit(X_train, y_train)
best_models['SVM'] = grid_svm.best_estimator_

# 3. Decision Tree
param_dt = {
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}
grid_dt = GridSearchCV(DecisionTreeClassifier(), param_dt, cv=5, n_jobs=-1)
grid_dt.fit(X_train, y_train)
best_models['DecisionTree'] = grid_dt.best_estimator_

# 4. Random Forest
param_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5]
}
grid_rf = GridSearchCV(RandomForestClassifier(), param_rf, cv=5, n_jobs=-1)
grid_rf.fit(X_train, y_train)
best_models['RandomForest'] = grid_rf.best_estimator_

# 5. Extra Trees
param_et = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10]
}
grid_et = GridSearchCV(ExtraTreesClassifier(), param_et, cv=5, n_jobs=-1)
grid_et.fit(X_train, y_train)
best_models['ExtraTrees'] = grid_et.best_estimator_

# 6. Gradient Boosting
param_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5]
}
grid_gb = GridSearchCV(GradientBoostingClassifier(), param_gb, cv=5, n_jobs=-1)
grid_gb.fit(X_train, y_train)
best_models['GradientBoost'] = grid_gb.best_estimator_

# 7. XGBoost
param_xgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 6]
}
grid_xgb = GridSearchCV(XGBClassifier(eval_metric='mlogloss'),
                        param_xgb, cv=5, n_jobs=-1)
grid_xgb.fit(X_train, y_train_numeric)
best_models['XGBoost'] = grid_xgb.best_estimator_

In [34]:
best_models

{'Logistic': LogisticRegression(C=1, max_iter=1000),
 'SVM': SVC(C=1, probability=True),
 'DecisionTree': DecisionTreeClassifier(),
 'RandomForest': RandomForestClassifier(min_samples_split=5),
 'ExtraTrees': ExtraTreesClassifier(),
 'GradientBoost': GradientBoostingClassifier(learning_rate=0.01, n_estimators=200),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='mlogloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.1, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=3, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=N

In [35]:
##Now working with tuned models
#Logistic Regression
grid_lr.best_estimator_
#grid_lr.best_estimator_.fit(X_train,y_train)
y_pred_lr=grid_lr.best_estimator_.predict(X_test)
##SVM
#grid_svm.best_estimator_.fit(X_train,y_train)
y_pred_svm=grid_svm.best_estimator_.predict(X_test)
#Decision Tree
#grid_dt.best_estimator_.fit(X_train,y_train)
y_pred_dt=grid_dt.best_estimator_.predict(X_test)
#Random Forest
#grid_rf.best_estimator_.fit(X_train,y_train)
y_pred_rf=grid_rf.best_estimator_.predict(X_test)
#Extra Trees
#grid_et.best_estimator_.fit(X_train,y_train)
y_pred_et=grid_et.best_estimator_.predict(X_test)
#Gradient Boosting
#grid_gb.best_estimator_.fit(X_train,y_train)
y_pred_gb=grid_gb.best_estimator_.predict(X_test)
#XGB
#grid_xgb.best_estimator_.fit(X_train,y_train_numeric)
y_pred_xgb=grid_xgb.best_estimator_.predict(X_test)

In [36]:
# 4. Check test set size
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Unique diseases in test: {y_test.nunique()}")

Training samples: 128
Testing samples: 32
Unique diseases in test: 16


In [38]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results = []

def evaluate_model(name, y_test, y_pred):
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1 Score": f1_score(y_test, y_pred, average='weighted')
    })

# Add all models
evaluate_model("Decision Tree", y_test, y_pred_dt)
evaluate_model("Random Forest", y_test, y_pred_rf)
evaluate_model("Extremely Randomized Trees", y_test, y_pred_et)
evaluate_model("Gradient Boosting", y_test, y_pred_gb)
evaluate_model("XGradient Boosting", y_test_numeric, y_pred_xgb)
evaluate_model("SVM", y_test, y_pred_svm)
#evaluate_model("SVM2", y_test, y_pred_svm2)
#evaluate_model("SVM3", y_test, y_pred_svm3)
evaluate_model("Logistic Regression", y_test, y_pred_lr)
# Create DataFrame
comparison_df = pd.DataFrame(results)

# Sort by Accuracy
comparison_df = comparison_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print(comparison_df)

# =========================
# Best Model Selection ⭐
# =========================

best_model = comparison_df.iloc[0]

print("\nBest Model:")
print(best_model)


                        Model  Accuracy  Precision   Recall  F1 Score
0                         SVM   1.00000   1.000000  1.00000  1.000000
1         Logistic Regression   1.00000   1.000000  1.00000  1.000000
2               Random Forest   0.96875   0.979167  0.96875  0.966667
3  Extremely Randomized Trees   0.90625   0.947917  0.90625  0.904167
4           Gradient Boosting   0.90625   0.947917  0.90625  0.904167
5          XGradient Boosting   0.90625   0.947917  0.90625  0.904167
6               Decision Tree   0.81250   0.895833  0.81250  0.808333

Best Model:
Model        SVM
Accuracy     1.0
Precision    1.0
Recall       1.0
F1 Score     1.0
Name: 0, dtype: object


In [41]:
print(grid_rf.best_score_)
print(grid_et.best_score_)  
print(grid_svm.best_score_) 
print(grid_lr.best_score_) 
print(grid_gb.best_score_)  
print(grid_xgb.best_score_) 
print(grid_dt.best_score_) 

0.960923076923077
0.952923076923077
0.9763076923076923
0.9686153846153847
0.8827692307692308
0.8824615384615384
0.8043076923076923


In [42]:
#Save the best model
import pickle

pickle.dump(grid_svm.best_estimator_, open("best_model.pkl", "wb"))

In [43]:
#Save Feature Names
import json

features = X.columns.tolist()

with open("features.json", "w") as f:
    json.dump(features, f)

In [44]:
#Test Prediction (Before Web)
model = pickle.load(open("best_model.pkl", "rb"))

sample = X_test.iloc[0]
print(model.predict([sample]))
print(model.predict_proba([sample]))

['Measles']
[[0.03665568 0.03858349 0.03516086 0.04174525 0.02626619 0.02762099
  0.0242078  0.03038089 0.03193488 0.0556414  0.0528193  0.4671744
  0.03771637 0.03408391 0.03635894 0.02364965]]


C:\Users\Ashutosh\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
C:\Users\Ashutosh\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(


In [45]:
pip install mysql-connector-python 

Note: you may need to restart the kernel to use updated packages.


In [46]:
import mysql.connector
db=mysql.connector.connect(
    host='localhost',
    user='root',
    password='Mysql@2441139',
    database='disease_app'
)
cursor=db.cursor()